In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

In [2]:
INPUT_PATH = '/media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/WMB-10Xv3-CB-raw.h5ad'
OUTPUT_PATH = '/media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/WMB-10Xv3-CB-raw.csv'
NUM_CELLS = 500

In [3]:
adata = sc.read_h5ad(INPUT_PATH)

In [4]:
matrix = adata.X.T
if sparse.issparse(matrix):
    matrix = matrix.toarray()
else:
    matrix = np.asarray(matrix)

cell_ids = adata.obs['cell_barcode'].index.astype(str).tolist() if 'cell_barcode' in adata.obs.columns else adata.obs_names.astype(str).tolist()

gene_symbols = adata.var['gene_symbol'].astype(str) if 'gene_symbol' in adata.var.columns else pd.Series(['nan_symbol'] * adata.n_vars)
var_ids = adata.var_names.astype(str)
combined_gene_ids = [f"{sym}" if sym not in (None, 'nan', 'nan_symbol') else gid for gid, sym in zip(var_ids, gene_symbols)]

matrix_df = pd.DataFrame(matrix, index=combined_gene_ids, columns=cell_ids)
matrix_df.index.name = None
matrix_df.reset_index(inplace=True)
matrix_df.rename(columns={'index': 'Unnamed: 0'}, inplace=True)

matrix_df.to_csv(OUTPUT_PATH, index=False)
print(f"CSV saved to {OUTPUT_PATH} with shape {matrix_df.shape}")

CSV saved to /media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/WMB-10Xv3-CB-raw.csv with shape (32285, 182027)


In [5]:
import pickle
with open("/media/ubuntu/sda/GeneCompass-main/prior_knowledge/human_mouse_tokens.pickle", 'rb') as f:
    a = pickle.load(f)